In [6]:
import pandas as pd
import numpy as np
import unicodedata 
import re

#cnpq_bruto = pd.read_csv("dados/cnpq_data_postgres.txt")
#cnpq_bruto.to_csv("dados/cnpq_data_postgres.csv", index=False)


### Função para normalizar os valores 

In [7]:
#
# Essa busca eu fiz no postgres, mas ela é mais eficiente no python
#
def parse_valor(valor):
    if pd.isna(valor):
        return np.nan
    
    valor = str(valor).strip()
    if not valor:
        return np.nan
    #
    # remove símbolos, alguns anos epecíficos possuem 'R$' ou '$' antes do número
    #
    valor = valor.replace('R$', '').replace('$', '').replace(' ', '')
    
    #
    # mantém apenas dígitos, vírgulas, pontos e sinais de menos
    #
    valor = ''.join(ch for ch in valor if ch.isdigit() or ch in ',.-')
    if valor in {'', '-', '.', ',', '-.', '-,'}:
        return np.nan

    ultima_virgula = valor.rfind(',')
    ultimo_ponto = valor.rfind('.')
    
    #
    # escolhe separador decimal pelo último símbolo encontrado
    #
    
    if ultima_virgula != -1 and ultimo_ponto != -1:
        if ultima_virgula > ultimo_ponto:
            normalizado = valor.replace('.', '').replace(',', '.')
        else:
            normalizado = valor.replace(',', '')
    elif ultima_virgula != -1:
        partes = valor.split(',')
        if len(partes[-1]) in (1, 2):
            normalizado = valor.replace('.', '').replace(',', '.')
        else:
            normalizado = valor.replace(',', '')
    elif ultimo_ponto != -1:
        partes = valor.split('.')
        if len(partes[-1]) in (1, 2):
            normalizado = valor.replace(',', '')
        else:
            normalizado = valor.replace('.', '')
    else:
        normalizado = valor

    try:
        return float(normalizado)
    except:
        return np.nan

### Aplica a função e exclui colunas categoricas desnecessárias 

In [8]:

cnpq_bruto = pd.read_csv("dados/cnpq_data_postgres.csv")

cnpq_bruto['valor_pago'] = cnpq_bruto['valor_pago'].apply(parse_valor)
cols_str = [
    'data_inicio_processo',
    'data_termino_processo',
    'regiao_destino',
    'titulo_do_projeto',
    'palavra_chave',
    'uo',
    'acaoppa',
    'cpf',
    'processo'
]

for col in cols_str:
    if col in cnpq_bruto.columns:
        cnpq_bruto[col] = cnpq_bruto[col].astype(str)

cols_para_excluir = [
    'data_extracao',
    'cod_modalidade',
    'id_lattes',
    'id_2020_ignorar'
    'acaoppa',
    'programappa',
    'cpf',
    'nome_chamada',
    'beneficiario',
    'processo',
    'data_inicio_processo',
    'data_termino_processo',
    'categoria_nivel',
    'uo',
    'natureza_de_despesa',

]

cnpq_bruto = cnpq_bruto.drop(columns=cols_para_excluir, errors='ignore')
cnpq_bruto = cnpq_bruto.drop(columns=['id_2020_ignorar','acaoppa'], errors='ignore')
cnpq_bruto.head()

C:\Users\felip\AppData\Local\Temp\ipykernel_8136\2018207577.py:1: DtypeWarning: Columns (0: data_inicio_processo, 1: data_termino_processo, 2: regiao_destino, 3: titulo_do_projeto, 4: palavra_chave, 5: uo, 6: valor_pago, 7: acaoppa, 8: cpf) have mixed types. Specify dtype option on import or set low_memory=False.
  cnpq_bruto = pd.read_csv("dados/cnpq_data_postgres.csv")


,_record_number,ano_referencia,linha_de_fomento,modalidade,programa_cnpq,grande_area,area,subarea,instituicao_origem,sigla_uf_origem,...,instituicao_destino,sigla_instituicao_destino,sigla_instituicao_macro,cidade_destino,sigla_uf_destino,regiao_destino,pais_destino,titulo_do_projeto,palavra_chave,valor_pago
0,3167657,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Institutos Nacionais de Ciência e Tec...,Ciências Exatas e da Terra,Química,Físico-Química,Universidade Federal de Minas Gerais,MG,...,Universidade Federal de Minas Gerais,UFMG,UFMG,Belo Horizonte,MG,SE,BRA - Brasil,INCT MIDAS. Tecnologias Ambientais Para a Valo...,materiais renováveis;tecnologias ambientais;re...,13750.0
1,3167658,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Especial de Cooperação com o Ministér...,Ciências da Saúde,Saúde Coletiva,Saúde Pública,SENAI - Departamento Regional da Bahia,BA,...,SENAI - Departamento Regional da Bahia,SENAI/DR/BA,SENAI/DR/BA,Salvador,BA,NE,BRA - Brasil,Estudos Clínicos de Fase I e II para a avaliaç...,COVID-19;LION;estudo clínico;nanocarreador lip...,2400.0
2,3167659,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Institutos Nacionais de Ciência e Tec...,Engenharias,Engenharia Biomédica,Bioengenharia,Universidade de Sao Paulo,SP,...,Universidade de Sao Paulo,USP,USP,Sao Paulo,SP,SE,BRA - Brasil,Instituto Nacional de Ciência e Tecnologia em ...,Processamento de imagens médicas;Modelagem e s...,19500.0
3,3167660,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Institutos Nacionais de Ciência e Tec...,Ciências Exatas e da Terra,Química,Físico-Química,Universidade Federal de Minas Gerais,MG,...,Universidade Federal de Minas Gerais,UFMG,UFMG,Belo Horizonte,MG,SE,BRA - Brasil,INCT MIDAS. Tecnologias Ambientais Para a Valo...,materiais renováveis;tecnologias ambientais;re...,3300.0
4,3167661,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,PROGRAMA DE AÇÕES VINCULADAS À BIOTECNOLOGIA E...,Ciências da Saúde,Medicina,Clínica Médica,Universidade de Sao Paulo,SP,...,Universidade de Sao Paulo,USP,USP,Sao Paulo,SP,SE,BRA - Brasil,Cardiomiocitos Derivados de iPSC humanas (hiPS...,regeneração cardíaca;terapia celular;célula tr...,7800.0


In [9]:
mapa_codigos = {

    # FORMAÇÃO 
    # Iniciação
    "IC": "Formação - Iniciação Científica",
    "ICJ": "Formação - Iniciação Científica",
    "ITI": "Formação - Iniciação Científica",
    "IT": "Formação - Iniciação Científica",
    "ITC": "Formação - Iniciação Científica",
    "IEX": "Formação - Iniciação Científica",

    # Mestrado
    "GM": "Formação - Mestrado",
    "MPE": "Formação - Mestrado",

    # Doutorado (Brasil)
    "GD": "Formação - Doutorado",

    # MOBILIDADE 
    # Doutorado sanduíche / exterior
    "SWE": "Mobilidade - Exterior",
    "SWI": "Mobilidade - Exterior",
    "SWP": "Mobilidade - Exterior",
    "GDE": "Mobilidade - Exterior",

    # Estágios / visitante / exterior geral
    "ESN": "Mobilidade - Exterior",
    "EJR": "Mobilidade - Exterior",
    "BSP": "Mobilidade - Exterior",
    "BEP": "Mobilidade - Exterior",
    "SPE": "Mobilidade - Exterior",
    "MSP": "Mobilidade - Exterior",
    "MEP": "Mobilidade - Exterior",

    # CARREIRA CIENTÍFICA 
    # Pós-doc
    "PD": "Carreira - Pós-Doutorado",
    "PDE": "Carreira - Pós-Doutorado",
    "PDJ": "Carreira - Pós-Doutorado",
    "PDS": "Carreira - Pós-Doutorado",
    "PDI": "Carreira - Pós-Doutorado",
    "PDT": "Carreira - Pós-Doutorado",
    "PDP": "Carreira - Pós-Doutorado",

    # Produtividade / pesquisador
    "PQ": "Carreira - Produtividade",
    "DT": "Carreira - Produtividade",

    # Fixação / pesquisador visitante
    "DCR": "Carreira - Fixação de Pesquisador",
    "RD": "Carreira - Fixação de Pesquisador",
    "PV": "Carreira - Fixação de Pesquisador",
    "PVE": "Carreira - Fixação de Pesquisador",

    # PESQUISA 
    # Projetos e financiamento direto
    "APQ": "Pesquisa - Auxílio à Pesquisa",
    "ARC": "Pesquisa - Auxílio à Pesquisa",
    "AED": "Pesquisa - Auxílio à Pesquisa",
    "ADC": "Pesquisa - Auxílio à Pesquisa",
    "AI": "Pesquisa - Auxílio à Pesquisa",

    # Apoio técnico 
    "AT": "Pesquisa - Apoio Técnico",
    "ATP": "Pesquisa - Apoio Técnico",

    # TECNOLOGIA & INOVAÇÃO 
    "DTI": "Tecnologia - Desenvolvimento",
    "DES": "Tecnologia - Desenvolvimento",
    "DEJ": "Tecnologia - Desenvolvimento",
    "DTC": "Tecnologia - Desenvolvimento",
    "SDT": "Tecnologia - Desenvolvimento",
    "INT": "Tecnologia - Desenvolvimento",
    "PIN": "Tecnologia - Desenvolvimento",

    # EVENTOS / DIFUSÃO 
    "AVG": "Difusão - Eventos Científicos",
    "EXP": "Difusão - Extensão",
    "EV": "Difusão - Visitante",
    "BEV": "Difusão - Visitante",

    # FALLBACK
    "SET": "Outros",
    "PCI": "Outros",
    "ACT": "Outros",
    "BJT": "Outros",
    "FIX": "Outros",
    "MAT": "Outros",
    "MDT": "Outros",
}

In [10]:
def normalizar(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto).upper()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    texto = re.sub(r'[^A-Z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto


def classificar_modalidade(mod):
    if pd.isna(mod):
        return "Não informado"

    mod_str = str(mod).upper()

    mod_norm = normalizar(mod_str)

    # FORMAÇÃO
    if "POS DOUTORADO" in mod_norm:
        return "Carreira - Pós-Doutorado"

    elif "DOUTORADO" in mod_norm:
        if "EXTERIOR" in mod_norm or "SANDUICHE" in mod_norm:
            return "Mobilidade - Doutorado - Exterior"
        return "Formação - Doutorado"

    elif "MESTRADO" in mod_norm:
        return "Formação - Mestrado"

    elif "INICIACAO" in mod_norm:
        return "Formação - Iniciação Científica"

    # CARREIRA
    elif "PRODUTIVIDADE" in mod_norm:
        return "Carreira - Produtividade"

    # TECNOLOGIA
    elif "DESENVOLVIMENTO" in mod_norm or "TECNOLOGICO" in mod_norm:
        return "Tecnologia - Desenvolvimento"

    # PESQUISA
    elif "APOIO" in mod_norm or "AUXILIO" in mod_norm:
        return "Pesquisa - Auxílio à Pesquisa"

    # MOBILIDADE 
    elif "EXTERIOR" in mod_norm or "VISITANTE" in mod_norm:
        return "Mobilidade - Exterior"


    return "Outros"


In [ ]:
cnpq_bruto['categoria'] = cnpq_bruto['modalidade'].apply(classificar_modalidade)
cnpq_bruto['ano_referencia'] = pd.to_numeric(cnpq_bruto['ano_referencia'], errors='coerce') 
cnpq_bruto['valor_pago'] = pd.to_numeric(cnpq_bruto['valor_pago'], errors='coerce')

print("\nCategorias criadas:")
print(cnpq_bruto['categoria'].value_counts())
cnpq_bruto.head()
#cnpq_bruto.drop(columns=['modalidade'], inplace=True)



Categorias criadas:
categoria
Formação - Iniciação Científica      1660539
Carreira - Produtividade              413774
Formação - Mestrado                   296404
Formação - Doutorado                  274407
Pesquisa - Auxílio à Pesquisa         240327
Outros                                119804
Tecnologia - Desenvolvimento          114632
Carreira - Pós-Doutorado               68035
Mobilidade - Exterior                  64015
Mobilidade - Doutorado - Exterior      22841
Name: count, dtype: int64


,_record_number,ano_referencia,linha_de_fomento,modalidade,programa_cnpq,grande_area,area,subarea,instituicao_origem,sigla_uf_origem,...,sigla_instituicao_destino,sigla_instituicao_macro,cidade_destino,sigla_uf_destino,regiao_destino,pais_destino,titulo_do_projeto,palavra_chave,valor_pago,categoria
0,3167657,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Institutos Nacionais de Ciência e Tec...,Ciências Exatas e da Terra,Química,Físico-Química,Universidade Federal de Minas Gerais,MG,...,UFMG,UFMG,Belo Horizonte,MG,SE,BRA - Brasil,INCT MIDAS. Tecnologias Ambientais Para a Valo...,materiais renováveis;tecnologias ambientais;re...,13750.0,Tecnologia - Desenvolvimento
1,3167658,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Especial de Cooperação com o Ministér...,Ciências da Saúde,Saúde Coletiva,Saúde Pública,SENAI - Departamento Regional da Bahia,BA,...,SENAI/DR/BA,SENAI/DR/BA,Salvador,BA,NE,BRA - Brasil,Estudos Clínicos de Fase I e II para a avaliaç...,COVID-19;LION;estudo clínico;nanocarreador lip...,2400.0,Tecnologia - Desenvolvimento
2,3167659,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Institutos Nacionais de Ciência e Tec...,Engenharias,Engenharia Biomédica,Bioengenharia,Universidade de Sao Paulo,SP,...,USP,USP,Sao Paulo,SP,SE,BRA - Brasil,Instituto Nacional de Ciência e Tecnologia em ...,Processamento de imagens médicas;Modelagem e s...,19500.0,Tecnologia - Desenvolvimento
3,3167660,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,Programa Institutos Nacionais de Ciência e Tec...,Ciências Exatas e da Terra,Química,Físico-Química,Universidade Federal de Minas Gerais,MG,...,UFMG,UFMG,Belo Horizonte,MG,SE,BRA - Brasil,INCT MIDAS. Tecnologias Ambientais Para a Valo...,materiais renováveis;tecnologias ambientais;re...,3300.0,Tecnologia - Desenvolvimento
4,3167661,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,PROGRAMA DE AÇÕES VINCULADAS À BIOTECNOLOGIA E...,Ciências da Saúde,Medicina,Clínica Médica,Universidade de Sao Paulo,SP,...,USP,USP,Sao Paulo,SP,SE,BRA - Brasil,Cardiomiocitos Derivados de iPSC humanas (hiPS...,regeneração cardíaca;terapia celular;célula tr...,7800.0,Tecnologia - Desenvolvimento


### Salva o arquivo -> parquet

In [12]:
import pyarrow as pa
#cnpq_bruto.to_parquet('dados/cnpqBolsasAuxilios.parquet', engine='pyarrow', index=False)
cnpq_bruto.to_parquet('dados/cnpqBolsasAuxilios.parquet', compression="zstd",  engine='pyarrow', index=False)
